# ISD Lite Weather Data Retrieval and Processing

This notebook uses the `pyisd` library to retrieve weather data from the ISD Lite dataset, processes it using Polars, and saves the results as a Parquet file for later use.

The format of the ISD Lite dataset is described in the [ISD Lite documentation](https://www.ncei.noaa.gov/pub/data/noaa/isd-lite/isd-lite-format.pdf). Each row in the dataset represents a single observation, and the columns include various weather parameters such as temperature, dew point, sea level pressure, wind direction, wind speed, sky condition, and precipitation.

The full ISD dataset does have some extra information we may want to include, such as ceiling and visibility, but for now we will just extract the information from the ISD Lite dataset for simplicity.

In [1]:
# Initialize the ISD Lite client
from pyisd import IsdLite

isd = IsdLite(crs=4326, verbose=True)

/Users/ozzysimpson/Git/Flight-Delay/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
STATION = "724050-13743"  # DCA
START = "2024-01-01"
END = "2025-08-26"  # This is the latest date for which there is ISD data

In [3]:
STATION = "724050-13743"
station_data = isd.get_data(
    start=START, end=END, station_id=STATION, organize_by="location"
)
weather_data = station_data[STATION]
weather_data.head()

100%|██████████| 1/1 [00:00<00:00,  2.38it/s]


,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,6.0,NaN,NaN
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,NaN,0.0,NaN
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,NaN,0.0,NaN
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,8.0,0.0,NaN
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,NaN,0.0,NaN


In [4]:
# Convert the data to Polars DataFrame, with the index as a column named 'timestamp'
import polars as pl

weather_df = pl.DataFrame(weather_data.reset_index())
weather_df = weather_df.rename({"index": "timestamp"})
weather_df.head()

timestamp,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h
datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,6.0,null,null
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,null,0.0,null
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,null,0.0,null
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,8.0,0.0,null
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,null,0.0,null


There are a few special cases to handle in the data:
- For precipitation, a value of -1 indicates a trace amount (less than 0.01 inches). We will create a new column to indicate whether the precipitation was a trace amount and replace the original -1 value with 0 for the precipitation amount.
- The sky coverage is represented by a code (e.g., 0 for clear, 1 for one okta, etc.). We'll convert the numeric codes into the actual labels for better readability.

In [5]:
# Replace trace precipitation (-1) with 0 and add a flag
weather_df = weather_df.with_columns(
    [
        # precipitation-1h_trace: 1 if original value was -1 (trace), 0 otherwise
        pl.when(pl.col("precipitation-1h") == -1.0)
        .then(1)
        .otherwise(0)
        .alias("precip-1h_trace"),
        # Replace -1 with 0 for the precipitation amount
        pl.when(pl.col("precipitation-1h") == -1.0)
        .then(0)
        .otherwise(pl.col("precipitation-1h"))
        .alias("precipitation-1h"),
        # precipitation-6h_trace: 1 if original value was -1 (trace), 0 otherwise
        pl.when(pl.col("precipitation-6h") == -1.0)
        .then(1)
        .otherwise(0)
        .alias("precip-6h_trace"),
        # Replace -1 with 0 for the precipitation amount
        pl.when(pl.col("precipitation-6h") == -1.0)
        .then(0)
        .otherwise(pl.col("precipitation-6h"))
        .alias("precipitation-6h"),
    ]
)

# Map numeric sky coverage codes to categorical labels
sky_coverage_map = {
    0: "clear",
    1: "1_okta",
    2: "2-3_oktas",
    3: "4_oktas",
    4: "5_oktas",
    5: "6_oktas",
    6: "7-8_oktas",
    7: "9+_oktas_not_full",
    8: "10_oktas_overcast",
    9: "obscured",
    10: "partial_obscured",
    11: "thin_scattered",
    12: "scattered",
    13: "dark_scattered",
    14: "thin_broken",
    15: "broken",
    16: "dark_broken",
    17: "thin_overcast",
    18: "overcast_standard",
    19: "dark_overcast",
    -9999: "missing",
}

# Create sky coverage label column
weather_df = weather_df.with_columns(
    [
        pl.col("skycoverage")
        .map_elements(
            lambda x: sky_coverage_map.get(x, "unknown"), return_dtype=pl.Utf8
        )
        .alias("skycoverage")
    ]
)

weather_df.head(100)

timestamp,temp,dewtemp,pressure,winddirection,windspeed,skycoverage,precipitation-1h,precipitation-6h,precip-1h_trace,precip-6h_trace
datetime[μs],f64,f64,f64,f64,f64,str,f64,f64,i32,i32
2024-01-01 00:00:00,7.2,-0.6,1016.3,160.0,3.1,"""7-8_oktas""",null,null,0,0
2024-01-01 01:00:00,6.7,0.0,1016.4,190.0,3.1,null,0.0,null,0,0
2024-01-01 02:00:00,6.7,0.0,1016.6,190.0,3.6,null,0.0,null,0,0
2024-01-01 03:00:00,6.7,0.6,1016.6,170.0,2.1,"""10_oktas_overcast""",0.0,null,0,0
2024-01-01 04:00:00,6.1,0.0,1016.1,130.0,2.6,null,0.0,null,0,0
…,…,…,…,…,…,…,…,…,…,…
2024-01-04 23:00:00,3.9,-7.2,1022.7,320.0,5.1,"""2-3_oktas""",0.0,null,0,0
2024-01-05 00:00:00,2.8,-7.2,1023.4,340.0,6.2,"""clear""",0.0,null,0,0
2024-01-05 01:00:00,2.2,-7.2,1024.0,320.0,5.1,"""clear""",0.0,null,0,0


In [6]:
# Save as parquet file
weather_df.write_parquet("isd_lite_weather_data.parquet")

In [7]:
# Write to database
from sqlalchemy import create_engine

# Create a SQLAlchemy engine for SQLite (or any other database)
engine = create_engine("sqlite:///isd_lite_weather_data.db")

weather_df.write_database(
    "weather_data", engine, engine="sqlalchemy", if_table_exists="replace"
)

14496